# 03 — Baseline Models & Temporal Validation

**Phase 4.** Before deploying any machine learning model, establishing rigorous statistical and heuristic baselines is mandatory.

### Principles
1. **Strict Temporal Split**: Future data must never leak into training. Split chronologically by date: 70% Train, 10% Validation, 20% Test.
2. **Standard Baselines**:
   - **Naive Forecast**: $\hat{y}_{t} = y_{t-1}$
   - **7-day Moving Average**: $\hat{y}_{t} = \frac{1}{7}\sum_{i=1}^7 y_{t-i}$
   - **14-day Moving Average**: $\hat{y}_{t} = \frac{1}{14}\sum_{i=1}^{14} y_{t-i}$
3. **Metric Selection**: MAE, RMSE, and WAPE (Weighted Absolute Percentage Error).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(
    p for p in (Path("."), Path(".."))
    if (p / "data" / "raw" / "observed_daily.csv").exists()
)
sys.path.insert(0, str(ROOT))

from src.data import load_observed, load_with_ground_truth
from src.features import build_features
from src.train import temporal_split, NaiveForecast, MovingAverageForecast
from src.evaluate import mae, rmse, wape, bias, evaluation_report, format_report

## 1. Load Data & Perform Chronological Split

In [ ]:
df_obs = load_observed()
df_truth = load_with_ground_truth()

# Build lag and calendar features
df_featured = build_features(df_obs)

# Merge ground truth strictly for evaluation comparison
eval_cols = ["date", "product", "true_demand", "is_censored", "lost_sales"]
df_eval = df_featured.merge(df_truth[eval_cols], on=["date", "product"], how="left")

train, val, test = temporal_split(df_eval, train_frac=0.70, val_frac=0.10)

print(f"Train : {len(train)} rows | {train['date'].min().date()} to {train['date'].max().date()}")
print(f"Val   : {len(val)} rows | {val['date'].min().date()} to {val['date'].max().date()}")
print(f"Test  : {len(test)} rows | {test['date'].min().date()} to {test['date'].max().date()}")

## 2. Fit and Evaluate Baselines on Validation Set

In [ ]:
baselines = {
    "Naive": NaiveForecast().fit(train),
    "MA-7": MovingAverageForecast(window=7).fit(train),
    "MA-14": MovingAverageForecast(window=14).fit(train),
}

val_results = []

for name, model in baselines.items():
    preds = model.predict(val)
    val_with_preds = val.copy()
    val_with_preds["pred"] = preds
    
    report = evaluation_report(
        val_with_preds,
        actual_col="true_demand",
        predicted_col="pred",
        censored_col="is_censored",
    )
    val_results.append({
        "Model": name,
        "MAE (All)": round(report["mae_all"], 3),
        "MAE (Censored)": round(report["mae_censored"], 3),
        "MAE (Uncensored)": round(report["mae_uncensored"], 3),
        "RMSE": round(report["rmse_all"], 3),
        "WAPE": round(report["wape_all"], 3),
        "Bias": round(report["bias_all"], 3),
    })

val_df = pd.DataFrame(val_results)
val_df

## 3. Visualizing Baseline Forecast vs True Demand

In [ ]:
# Inspect Milk predictions on Validation set
milk_val = val[val["product"] == "Milk"].copy()
milk_val["pred_naive"] = baselines["Naive"].predict(milk_val)
milk_val["pred_ma7"] = baselines["MA-7"].predict(milk_val)

plt.figure(figsize=(12, 5))
plt.plot(milk_val["date"], milk_val["true_demand"], label="True Demand", color="black", linewidth=2)
plt.plot(milk_val["date"], milk_val["sales"], label="Recorded Sales", color="gray", linestyle=":", linewidth=1.5)
plt.plot(milk_val["date"], milk_val["pred_naive"], label="Naive (Lag 1)", color="#e74c3c", linestyle="--", alpha=0.7)
plt.plot(milk_val["date"], milk_val["pred_ma7"], label="7-Day Moving Average", color="#4f8ef7", linewidth=2)

plt.title("Milk Validation Set: Baselines vs True Demand", fontsize=13)
plt.ylabel("Units")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Baseline Takeaways

- **7-Day Moving Average** offers the strongest baseline with significantly lower variance than Naive (Lag 1).
- All standard baselines exhibit negative bias during censored periods because they average over past recorded sales (which were constrained by empty shelves).